# Extracción de Entidades

## *boltuix NeuroBERT-NER*

Para la tarea de detección de entidades vamos a utilizar los embeddings por subtoken (aplicando last-4-layers mean) generados previamente con el modelo *boltuix NeuroBERT-NER*. El siguiente paso es aplicar la cabeza de clasificación de NER (la capa lineal del modelo) sobre cada embedding de subtoken para obtener las etiquetas BIO o *logits* y luego agrupar spans.

In [42]:
from transformers import AutoModelForTokenClassification, AutoConfig, AutoTokenizer
import pandas as pd

model_name = "boltuix/NeuroBERT-NER"
config = AutoConfig.from_pretrained(model_name)
id2label = config.id2label
label2id = config.label2id

print("=" * 50)
print(f"Total de etiquetas: {len(id2label)}")
print("=" * 50)
print(id2label)

Total de etiquetas: 37
{0: 'B-CARDINAL', 1: 'B-DATE', 2: 'B-EVENT', 3: 'B-FAC', 4: 'B-GPE', 5: 'B-LANGUAGE', 6: 'B-LAW', 7: 'B-LOC', 8: 'B-MONEY', 9: 'B-NORP', 10: 'B-ORDINAL', 11: 'B-ORG', 12: 'B-PERCENT', 13: 'B-PERSON', 14: 'B-PRODUCT', 15: 'B-QUANTITY', 16: 'B-TIME', 17: 'B-WORK_OF_ART', 18: 'I-CARDINAL', 19: 'I-DATE', 20: 'I-EVENT', 21: 'I-FAC', 22: 'I-GPE', 23: 'I-LANGUAGE', 24: 'I-LAW', 25: 'I-LOC', 26: 'I-MONEY', 27: 'I-NORP', 28: 'I-ORDINAL', 29: 'I-ORG', 30: 'I-PERCENT', 31: 'I-PERSON', 32: 'I-PRODUCT', 33: 'I-QUANTITY', 34: 'I-TIME', 35: 'I-WORK_OF_ART', 36: 'O'}


Vamos a cargar los tokens generados con el mismo modelo para la deteccion de entidades

In [35]:
tokens = pd.read_parquet("../Representacion_del_lenguaje/datasPost_prepro/fin-bertTokenized.parquet")
print(tokens.head(3))

                                        article_text  \
0  The UK jobs market continues to show signs of ...   
1  New data from the Department from Work and Pen...   
2  Asking for workplace accommodations is often e...   

                                           input_ids  \
0  [101, 1996, 2866, 5841, 3006, 4247, 2000, 2265...   
1  [101, 2047, 2951, 2013, 1996, 2533, 2013, 2147...   
2  [101, 4851, 2005, 16165, 26167, 2003, 2411, 60...   

                                      attention_mask  
0  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...  
1  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...  
2  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...  


In [36]:
tokens = pd.read_parquet("../Representacion_del_lenguaje/datasPost_prepro/fin-bertTokenized.parquet")
article_text = tokens.loc[:,"article_text"]
input_ids = tokens.loc[:,"input_ids"]
attention_mask = tokens.loc[:,"attention_mask"]

print(article_text[0][0:96], "...")
print(input_ids[0])
print(attention_mask[0])

The UK jobs market continues to show signs of weakness, with pay growth slowing and unemployment ...
[101, 1996, 2866, 5841, 3006, 4247, 2000, 2265, 5751, 1997, 11251, 1010, 2007, 3477, 3930, 18068, 1998, 12163, 3968, 4726, 3020, 3805, 1997, 1996, 7114, 5166, 2279, 3204, 1012, 1996, 6745, 2951, 2013, 1996, 2436, 2005, 2120, 6747, 1006, 2006, 2015, 1007, 1010, 2207, 2006, 9857, 1010, 3662, 2008, 3296, 11897, 3930, 13343, 29563, 1999, 1996, 2093, 2706, 2000, 2257, 2001, 1018, 1012, 1021, 1003, 1010, 2091, 3621, 2013, 1018, 1012, 1022, 1003, 2090, 2089, 1998, 2251, 1012, 1996, 12163, 3446, 2234, 1999, 2012, 1018, 1012, 1022, 1003, 2005, 1996, 2558, 1010, 3621, 3020, 2084, 1996, 1018, 1012, 1021, 1003, 2680, 2005, 1996, 3025, 2093, 2706, 1012, 1996, 2193, 1997, 5126, 2006, 1996, 26854, 1999, 1996, 2095, 2000, 2257, 2001, 4358, 2000, 2031, 5357, 2011, 6109, 1010, 2199, 1010, 2295, 2009, 3445, 2011, 2184, 1010, 2199, 2090, 2251, 1998, 2257, 1012, 2220, 10035, 2005, 1996, 2193, 1997, 26854, 2

In [37]:
import torch

try:
    # Convertir las listas a tensores de PyTorch
    t_input_ids = torch.tensor(input_ids)
    t_attention_mask = torch.tensor(attention_mask)

    print("Datos cargados con fastparquet.")
    print("input_ids:\n", t_input_ids)
    print("attention_mask:\n", t_attention_mask)
except Exception as e:
    print(f"Error al usar fastparquet: {e}")

Datos cargados con fastparquet.
input_ids:
 tensor([[  101,  1996,  2866,  ...,     0,     0,     0],
        [  101,  2047,  2951,  ...,  2077,  2037,   102],
        [  101,  4851,  2005,  ...,  5005,  1011,   102],
        ...,
        [  101,  7211,  1006,  ...,     0,     0,     0],
        [  101,  5148,  1521,  ..., 29336,  5369,   102],
        [  101,  2054,  2064,  ...,  2028,  1011,   102]])
attention_mask:
 tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1]])


In [38]:
model = AutoModelForTokenClassification.from_pretrained(model_name,
                                                        num_labels=len(id2label),
                                                        id2label=id2label,
                                                        label2id=label2id)

Para esta tarea no necesitamos que en el output se añadan los hidden states, ya que el objetivo es usar la cabeza NER original, sin aplicar el layer pooling, por lo que lo ideal es usar el *outputs.logits* directamente.

In [43]:
sample_text = article_text[0]
sample_input_ids = t_input_ids[0].unsqueeze(0)  # Añadir dimensión de batch [1, seq_len]
sample_attention_mask = t_attention_mask[0].unsqueeze(0)  # Añadir dimensión de batch [1, seq_len]

with torch.no_grad():
    outputs = model(input_ids=sample_input_ids, attention_mask=sample_attention_mask)

predictions = outputs.logits.argmax(dim=-1)[0].cpu().numpy()
tokens = AutoTokenizer.from_pretrained("boltuix/NeuroBERT-NER").convert_ids_to_tokens(sample_input_ids[0])
word_ids = sample_input_ids[0].cpu().numpy()


In [50]:
# Visualización de tokens con sus etiquetas BIO
print("=" * 60)
print("ENTIDADES DETECTADAS (Etiquetas B- e I-)")
print("=" * 60)

entity_count = 0
for idx, (token, pred_id) in enumerate(zip(tokens, predictions)):
    label = model.config.id2label[pred_id]
    
    # Solo mostrar tokens que tienen etiquetas B- o I- (entidades)
    if label.startswith('B-') or label.startswith('I-'):
        print(f"Token: {token:25} | Label: {label}")
        entity_count += 1

print("=" * 60)
print(f"Total de tokens clasificados como entidad: {entity_count}")
print("=" * 60)


ENTIDADES DETECTADAS (Etiquetas B- e I-)
Token: uk                        | Label: B-ORG
Token: the                       | Label: B-DATE
Token: autumn                    | Label: B-DATE
Token: next                      | Label: I-DATE
Token: month                     | Label: I-DATE
Token: ##s                       | Label: B-ORG
Token: tuesday                   | Label: B-DATE
Token: the                       | Label: B-DATE
Token: three                     | Label: B-DATE
Token: months                    | Label: I-DATE
Token: august                    | Label: B-DATE
Token: 4                         | Label: B-PERCENT
Token: .                         | Label: B-PERCENT
Token: 7                         | Label: B-PERCENT
Token: %                         | Label: I-PERCENT
Token: 4                         | Label: B-PERCENT
Token: .                         | Label: B-PERCENT
Token: 8                         | Label: B-PERCENT
Token: %                         | Label: I-PERCENT
Token: